# Análise do Choque Geopolítico e Resultados

Este notebook reúne as análises construídas a partir das tabelas da camada Gold para investigar o comportamento do mercado de combustíveis em torno da escalada do conflito no Oriente Médio em 2026.

A análise busca responder às seguintes questões:

- Como o preço internacional do petróleo Brent se comportou antes e depois do evento?
- Como evoluíram os preços de paridade de importação (PPI) de gasolina e diesel?
- As oscilações internacionais apareceram imediatamente nos preços ao consumidor brasileiro?
- Gasolina e diesel apresentaram comportamentos diferentes?
- Existe defasagem temporal entre os movimentos do Brent/PPI e os preços ao consumidor?
- O comportamento dos preços foi diferente entre estados e regiões brasileiras?

## Contexto do evento

Para comparar o comportamento das séries antes e depois da escalada do conflito, foi definida uma data de referência com base na cronologia dos acontecimentos. A partir dela, foram construídas janelas equivalentes de 12 semanas anteriores e posteriores ao evento.

In [0]:
brent_gold = spark.table("workspace.gold.brent_semanal")
ppi_gold = spark.table("workspace.gold.ppi_semanal")
anp_gold = spark.table("workspace.gold.anp_semanal")
base_integrada = spark.table("workspace.gold.base_integrada")
corr_brent_ppi = spark.table("workspace.gold.correlacao_brent_ppi")
corr_anp = spark.table("workspace.gold.correlacoes_anp")

print("Tabelas Gold carregadas com sucesso.")

Tabelas Gold carregadas com sucesso.


## Definição da data de referência

Foi adotado o dia **28 de fevereiro de 2026** como data de referência para a análise.

A escolha foi feita a partir da cronologia dos acontecimentos, antes de considerar o comportamento das séries de preços. Naquela data ocorreu uma forte escalada do conflito no Oriente Médio, com ataques dos Estados Unidos e de Israel contra o Irã e ataques iranianos subsequentes na região.

O período também foi marcado por restrições ao tráfego no Estreito de Ormuz e impactos sobre a produção e o transporte de petróleo, com repercussões no mercado internacional de energia.

Como os dados utilizados no projeto têm granularidade semanal, a semana iniciada em 23/02/2026 foi considerada a última do período anterior, enquanto 02/03/2026 marca a primeira semana completa posterior ao evento. Para a comparação foram utilizadas duas janelas simétricas de 12 semanas.

A data funciona apenas como marco temporal. A comparação antes e depois não pressupõe que todas as variações posteriores tenham sido provocadas pelo conflito, já que os preços de petróleo e combustíveis respondem simultaneamente a diversos fatores.

In [0]:
from pyspark.sql.functions import col, lit, datediff, abs as spark_abs

data_evento = "2026-02-28"

print("BRENT")
display(
    brent_gold
    .withColumn(
        "distancia_evento",
        spark_abs(datediff(col("semana"), lit(data_evento)))
    )
    .orderBy("distancia_evento")
    .select(
        "semana",
        "brent_medio_usd",
        "brent_min_usd",
        "brent_max_usd",
        "dias_observados"
    )
    .limit(6)
)

print("PPI")
display(
    ppi_gold
    .withColumn(
        "distancia_evento",
        spark_abs(datediff(col("data_inicio"), lit(data_evento)))
    )
    .orderBy("distancia_evento", "produto")
    .select(
        "data_inicio",
        "produto",
        "ppi_medio",
        "ppi_min",
        "ppi_max"
    )
    .limit(12)
)

print("ANP")
display(
    anp_gold
    .withColumn(
        "distancia_evento",
        spark_abs(datediff(col("semana"), lit(data_evento)))
    )
    .orderBy("distancia_evento", "produto")
    .select(
        "semana",
        "produto",
        "preco_medio_anp"
    )
    .limit(12)
)

BRENT


semana,brent_medio_usd,brent_min_usd,brent_max_usd,dias_observados
2026-03-02,85.282,77.24,95.74,5
2026-02-23,71.35600000000001,70.69,71.9,5
2026-03-09,96.156,89.84,103.23,5
2026-02-16,71.65599999999999,69.77,73.17,5
2026-03-16,111.398,101.04,118.42,5
2026-02-09,70.696,69.8,71.52,5


PPI


data_inicio,produto,ppi_medio,ppi_min,ppi_max
2026-03-02,DIESEL,4.645515250000001,4.471640000000001,4.806292
2026-03-02,GASOLINA,2.965749375,2.866142,3.115966
2026-02-23,DIESEL,3.479925125,3.362668,3.634528
2026-02-23,GASOLINA,2.5562134999999997,2.474056,2.704532
2026-03-09,DIESEL,5.319894875,5.136284,5.48158
2026-03-09,GASOLINA,3.4589377499999996,3.360018,3.6088180000000003
2026-02-16,DIESEL,3.3602955000000003,3.249378,3.51628
2026-02-16,GASOLINA,2.51190475,2.429795,2.662295
2026-03-16,DIESEL,6.00836125,5.8037160000000005,6.171608
2026-03-16,GASOLINA,3.9201992500000005,3.8143119999999997,4.070138


ANP


semana,produto,preco_medio_anp
2026-03-02,DIESEL S10,6.183609348914857
2026-03-02,GASOLINA,6.307111368909509
2026-02-23,DIESEL S10,6.138680808750412
2026-02-23,GASOLINA,6.300219197046604
2026-03-09,DIESEL S10,6.863787043418331
2026-03-09,GASOLINA,6.481970349115254
2026-02-16,DIESEL S10,6.148176609369734
2026-02-16,GASOLINA,6.310519055412668
2026-03-16,DIESEL S10,7.342990463215255
2026-03-16,GASOLINA,6.667955545265103


In [0]:
from pyspark.sql.functions import (
    col, lit, when, avg, min, max, count
)

inicio_pre = "2025-12-08"
fim_pre = "2026-02-23"

inicio_pos = "2026-03-02"
fim_pos = "2026-05-18"

brent_evento = (
    brent_gold
    .filter(
        ((col("semana") >= lit(inicio_pre)) & (col("semana") <= lit(fim_pre))) |
        ((col("semana") >= lit(inicio_pos)) & (col("semana") <= lit(fim_pos)))
    )
    .withColumn(
        "periodo",
        when(col("semana") <= lit(fim_pre), "Pré-choque")
        .otherwise("Pós-choque")
    )
)

resumo_brent_evento = (
    brent_evento
    .groupBy("periodo")
    .agg(
        avg("brent_medio_usd").alias("preco_medio_usd"),
        min("brent_medio_usd").alias("menor_media_semanal_usd"),
        max("brent_medio_usd").alias("maior_media_semanal_usd"),
        count("*").alias("semanas")
    )
    .orderBy("periodo")
)

display(resumo_brent_evento)

periodo,preco_medio_usd,menor_media_semanal_usd,maior_media_semanal_usd,semanas
Pré-choque,66.64176388888889,60.826,71.65599999999999,12
Pós-choque,110.27612500000002,85.282,124.605,12


In [0]:
from pyspark.sql.functions import (
    col, round as spark_round
)

# Médias dos períodos pré e pós-choque
medias_brent = {
    row["periodo"]: row["preco_medio_usd"]
    for row in resumo_brent_evento.collect()
}

media_pre = medias_brent["Pré-choque"]
media_pos = medias_brent["Pós-choque"]

variacao_brent_pct = (
    (media_pos - media_pre) / media_pre
) * 100

print(f"Média pré-choque: US$ {media_pre:.2f}")
print(f"Média pós-choque: US$ {media_pos:.2f}")
print(f"Variação entre as médias: {variacao_brent_pct:.2f}%")

print("\nSemana de maior preço médio no período pós-choque:")

display(
    brent_evento
    .filter(col("periodo") == "Pós-choque")
    .select(
        "semana",
        "brent_medio_usd",
        "brent_min_usd",
        "brent_max_usd"
    )
    .orderBy(col("brent_medio_usd").desc())
    .limit(1)
)

Média pré-choque: US$ 66.64
Média pós-choque: US$ 110.28
Variação entre as médias: 65.48%

Semana de maior preço médio no período pós-choque:


semana,brent_medio_usd,brent_min_usd,brent_max_usd
2026-04-06,124.605,119.03,138.21


In [0]:
brent_evento_grafico = (
    brent_evento
    .select(
        "semana",
        "brent_medio_usd",
        "periodo"
    )
    .orderBy("semana")
)

display(brent_evento_grafico)

semana,brent_medio_usd,periodo
2025-12-08,62.604,Pré-choque
2025-12-15,60.826,Pré-choque
2025-12-22,63.20666666666667,Pré-choque
2025-12-29,62.1825,Pré-choque
2026-01-05,62.926,Pré-choque
2026-01-12,66.99600000000001,Pré-choque
2026-01-19,66.98599999999999,Pré-choque
2026-01-26,70.426,Pré-choque
2026-02-02,69.84,Pré-choque
2026-02-09,70.696,Pré-choque


Databricks visualization. Run in Databricks to view.

### Comportamento do Brent antes e depois do choque

Na janela de 12 semanas anteriores à data de referência, o preço médio semanal do petróleo Brent foi de **US$ 66,64 por barril**, com médias semanais variando entre US$ 60,83 e US$ 71,66.

Nas 12 semanas posteriores, a média aumentou para **US$ 110,28 por barril**, representando uma elevação de **65,48%** em relação à média do período anterior.

A elevação ocorreu rapidamente após a escalada do conflito. A média semanal passou de aproximadamente **US$ 71,36** na semana iniciada em 23/02/2026 para **US$ 85,28** em 02/03 e **US$ 96,16** em 09/03.

O maior valor médio semanal dentro da janela pós-choque foi registrado na semana iniciada em **06/04/2026**, quando o Brent atingiu média de **US$ 124,61 por barril**.

In [0]:
from pyspark.sql.functions import (
    col, lit, when, avg, min, max, count
)

ppi_evento = (
    ppi_gold
    .filter(
        (
            (col("data_inicio") >= lit(inicio_pre)) &
            (col("data_inicio") <= lit(fim_pre))
        ) |
        (
            (col("data_inicio") >= lit(inicio_pos)) &
            (col("data_inicio") <= lit(fim_pos))
        )
    )
    .withColumn(
        "periodo",
        when(
            col("data_inicio") <= lit(fim_pre),
            "Pré-choque"
        ).otherwise("Pós-choque")
    )
)

resumo_ppi_evento = (
    ppi_evento
    .groupBy("produto", "periodo")
    .agg(
        avg("ppi_medio").alias("ppi_medio_periodo"),
        min("ppi_medio").alias("menor_media_semanal"),
        max("ppi_medio").alias("maior_media_semanal"),
        count("*").alias("semanas")
    )
    .orderBy("produto", "periodo")
)

display(resumo_ppi_evento)

produto,periodo,ppi_medio_periodo,menor_media_semanal,maior_media_semanal,semanas
DIESEL,Pré-choque,3.2579896510416666,3.0625737499999994,3.479925125,12
DIESEL,Pós-choque,5.641180833333333,4.645515250000001,6.400695749999999,12
GASOLINA,Pré-choque,2.424117151041666,2.2645999999999997,2.5562134999999997,12
GASOLINA,Pós-choque,3.980887364583334,2.965749375,4.442294749999999,12


In [0]:
from pyspark.sql.functions import (
    col,
    max as spark_max,
    round as spark_round
)

comparacao_ppi = (
    resumo_ppi_evento
    .groupBy("produto")
    .pivot("periodo", ["Pré-choque", "Pós-choque"])
    .agg(spark_max("ppi_medio_periodo"))
    .withColumn(
        "variacao_absoluta",
        col("Pós-choque") - col("Pré-choque")
    )
    .withColumn(
        "variacao_percentual",
        (
            (col("Pós-choque") - col("Pré-choque"))
            / col("Pré-choque")
        ) * 100
    )
    .select(
        "produto",
        spark_round(col("Pré-choque"), 3).alias("media_pre"),
        spark_round(col("Pós-choque"), 3).alias("media_pos"),
        spark_round(col("variacao_absoluta"), 3).alias("variacao_absoluta"),
        spark_round(col("variacao_percentual"), 2).alias("variacao_percentual")
    )
    .orderBy("produto")
)

display(comparacao_ppi)

produto,media_pre,media_pos,variacao_absoluta,variacao_percentual
DIESEL,3.258,5.641,2.383,73.15
GASOLINA,2.424,3.981,1.557,64.22


In [0]:
ppi_evento_grafico = (
    ppi_evento
    .select(
        "data_inicio",
        "produto",
        "ppi_medio",
        "periodo"
    )
    .orderBy("data_inicio", "produto")
)

display(ppi_evento_grafico)

data_inicio,produto,ppi_medio,periodo
2025-12-08,DIESEL,3.2839869999999998,Pré-choque
2025-12-08,GASOLINA,2.45123325,Pré-choque
2025-12-15,DIESEL,3.1447318749999997,Pré-choque
2025-12-15,GASOLINA,2.32824075,Pré-choque
2025-12-22,DIESEL,3.16966625,Pré-choque
2025-12-22,GASOLINA,2.353425125,Pré-choque
2025-12-29,DIESEL,3.1367893125000004,Pré-choque
2025-12-29,GASOLINA,2.3248611250000004,Pré-choque
2026-01-05,DIESEL,3.0625737499999994,Pré-choque
2026-01-05,GASOLINA,2.2645999999999997,Pré-choque


Databricks visualization. Run in Databricks to view.

### Comportamento do PPI de gasolina e diesel

Os preços de paridade de importação apresentaram elevação relevante após a data de referência adotada para a escalada do conflito.

Para o **diesel**, o PPI médio passou de **3,258** nas 12 semanas anteriores para **5,641** nas 12 semanas posteriores, correspondendo a uma elevação de **73,15%** entre as médias dos dois períodos.

Para a **gasolina**, a média passou de **2,424** para **3,981**, uma elevação de **64,22%**.

Dessa forma, ambos os combustíveis apresentaram aumento expressivo do PPI na janela analisada. A variação percentual observada foi maior para o diesel, cuja elevação entre as médias pré e pós-choque superou a da gasolina em **8,93 pontos percentuais**.

No período analisado, portanto, o PPI do diesel apresentou uma alta maior que o da gasolina. A comparação com as correlações históricas, apresentada adiante, ajuda a verificar se essa diferença também aparece fora do episódio específico.

## Repasse dos preços dos combustíveis ao consumidor

Após analisar o comportamento do Brent e dos preços de paridade de importação, esta etapa avalia se as mudanças observadas no Brent e no PPI foram acompanhadas por alterações imediatas nos preços de gasolina e diesel ao consumidor brasileiro.

Para manter a comparabilidade com as análises anteriores, são utilizadas as mesmas janelas de 12 semanas anteriores e posteriores à data de referência.

In [0]:
from pyspark.sql.functions import (
    col, lit, when, avg, min, max, count
)

anp_evento = (
    anp_gold
    .filter(
        (
            (col("semana") >= lit(inicio_pre)) &
            (col("semana") <= lit(fim_pre))
        ) |
        (
            (col("semana") >= lit(inicio_pos)) &
            (col("semana") <= lit(fim_pos))
        )
    )
    .withColumn(
        "periodo",
        when(
            col("semana") <= lit(fim_pre),
            "Pré-choque"
        ).otherwise("Pós-choque")
    )
)

resumo_anp_evento = (
    anp_evento
    .groupBy("produto", "periodo")
    .agg(
        avg("preco_medio_anp").alias("preco_medio_periodo"),
        min("preco_medio_anp").alias("menor_media_semanal"),
        max("preco_medio_anp").alias("maior_media_semanal"),
        count("*").alias("semanas")
    )
    .orderBy("produto", "periodo")
)

display(resumo_anp_evento)

produto,periodo,preco_medio_periodo,menor_media_semanal,maior_media_semanal,semanas
DIESEL S10,Pré-choque,6.143286937189962,6.109610067352002,6.1711479850999,12
DIESEL S10,Pós-choque,7.253981771154874,6.183609348914857,7.583704446673351,12
GASOLINA,Pré-choque,6.282081701274819,6.195339066339063,6.336649472450175,12
GASOLINA,Pós-choque,6.679182063219714,6.307111368909509,6.808806106174523,12


In [0]:
from pyspark.sql.functions import (
    col,
    max as spark_max,
    round as spark_round
)

comparacao_anp = (
    resumo_anp_evento
    .groupBy("produto")
    .pivot("periodo", ["Pré-choque", "Pós-choque"])
    .agg(spark_max("preco_medio_periodo"))
    .withColumn(
        "variacao_absoluta",
        col("Pós-choque") - col("Pré-choque")
    )
    .withColumn(
        "variacao_percentual",
        (
            (col("Pós-choque") - col("Pré-choque"))
            / col("Pré-choque")
        ) * 100
    )
    .select(
        "produto",
        spark_round(col("Pré-choque"), 3).alias("media_pre"),
        spark_round(col("Pós-choque"), 3).alias("media_pos"),
        spark_round(col("variacao_absoluta"), 3).alias("variacao_absoluta"),
        spark_round(col("variacao_percentual"), 2).alias("variacao_percentual")
    )
    .orderBy("produto")
)

display(comparacao_anp)

produto,media_pre,media_pos,variacao_absoluta,variacao_percentual
DIESEL S10,6.143,7.254,1.111,18.08
GASOLINA,6.282,6.679,0.397,6.32


In [0]:
from pyspark.sql.functions import col, lag, round as spark_round
from pyspark.sql.window import Window

w_anp = Window.partitionBy("produto").orderBy("semana")

anp_reacao_semanal = (
    anp_gold
    .filter(
        (col("semana") >= lit("2026-02-09")) &
        (col("semana") <= lit("2026-03-30"))
    )
    .withColumn(
        "preco_semana_anterior",
        lag("preco_medio_anp").over(w_anp)
    )
    .withColumn(
        "variacao_semanal_pct",
        (
            (col("preco_medio_anp") - col("preco_semana_anterior"))
            / col("preco_semana_anterior")
        ) * 100
    )
    .select(
        "semana",
        "produto",
        spark_round("preco_medio_anp", 3).alias("preco_medio"),
        spark_round("variacao_semanal_pct", 2).alias("variacao_semanal_pct")
    )
    .orderBy("semana", "produto")
)

display(anp_reacao_semanal)

semana,produto,preco_medio,variacao_semanal_pct
2026-02-09,DIESEL S10,6.157,null
2026-02-09,GASOLINA,6.314,null
2026-02-16,DIESEL S10,6.148,-0.14
2026-02-16,GASOLINA,6.311,-0.06
2026-02-23,DIESEL S10,6.139,-0.15
2026-02-23,GASOLINA,6.3,-0.16
2026-03-02,DIESEL S10,6.184,0.73
2026-03-02,GASOLINA,6.307,0.11
2026-03-09,DIESEL S10,6.864,11.0
2026-03-09,GASOLINA,6.482,2.77


In [0]:
anp_evento_grafico = (
    anp_evento
    .select(
        "semana",
        "produto",
        "preco_medio_anp"
    )
    .orderBy("semana", "produto")
)

display(anp_evento_grafico)

semana,produto,preco_medio_anp
2025-12-08,DIESEL S10,6.1177854046242786
2025-12-08,GASOLINA,6.195339066339063
2025-12-15,DIESEL S10,6.110843330980946
2025-12-15,GASOLINA,6.199245783132525
2025-12-22,DIESEL S10,6.109610067352002
2025-12-22,GASOLINA,6.2161735315622675
2025-12-29,DIESEL S10,6.125376425855513
2025-12-29,GASOLINA,6.228107966457021
2026-01-05,DIESEL S10,6.15208025343189
2026-01-05,GASOLINA,6.297115902964955


Databricks visualization. Run in Databricks to view.

### Repasse dos preços ao consumidor

Os preços médios ao consumidor apresentaram elevação após a data de referência, porém em intensidade e velocidade diferentes das observadas no Brent e no PPI.

Na comparação entre as janelas de 12 semanas, o preço médio do **Diesel S10** passou de R$ 6,143 para R$ 7,254, uma elevação de **18,08%**. Para a **gasolina**, a média passou de R$ 6,282 para R$ 6,679, correspondendo a **6,32%**.

A análise semanal mostra que a maior parte do movimento não apareceu na primeira semana posterior ao evento. Na semana iniciada em 02/03/2026, o preço médio do Diesel S10 aumentou **0,73%**, enquanto o da gasolina apresentou variação de apenas **0,11%**. Na semana seguinte, iniciada em 09/03, as variações foram de **11,00%** para o Diesel S10 e **2,77%** para a gasolina.

O movimento prosseguiu na semana de 16/03, com novas altas de **6,98%** no Diesel S10 e **2,87%** na gasolina.

Esses resultados são consistentes com a análise de correlação com defasagens realizada anteriormente. Na janela comum, as maiores correlações entre as variações do Brent/PPI e dos preços ao consumidor foram observadas com **uma semana de defasagem (lag 1)** para ambos os combustíveis.

Portanto, os dados não indicam um repasse integral e contemporâneo das oscilações internacionais aos preços ao consumidor na primeira semana após o evento. Na janela analisada, a reação dos preços domésticos tornou-se mais pronunciada nas semanas seguintes, especialmente para o Diesel S10.

## Sensibilidade da gasolina e do diesel e defasagem temporal

Os resultados indicam comportamentos distintos entre gasolina e diesel, dependendo da dimensão analisada.

Na comparação das 12 semanas anteriores e posteriores à data de referência, o **PPI do diesel apresentou aumento de 73,15%**, enquanto o **PPI da gasolina aumentou 64,22%**. Nos preços ao consumidor, a diferença foi ainda mais acentuada: o preço médio do **Diesel S10 aumentou 18,08%**, enquanto o da **gasolina aumentou 6,32%**.

Durante as primeiras semanas posteriores ao evento, o Diesel S10 também apresentou variações semanais mais intensas. Na semana iniciada em 09/03/2026, por exemplo, o preço médio do diesel aumentou 11,00%, frente a 2,77% da gasolina.

Entretanto, a análise histórica da relação entre Brent e PPI apresenta uma perspectiva complementar. Entre 2018 e 2026, a correlação contemporânea entre as variações do Brent e do PPI foi de aproximadamente **0,588 para o diesel** e **0,690 para a gasolina**. Assim, embora o diesel tenha apresentado maior aumento percentual durante o choque analisado, a associação histórica contemporânea entre Brent e PPI foi mais forte para a gasolina.

Dessa forma, não há uma única medida capaz de definir de forma geral qual combustível é estruturalmente mais sensível às oscilações internacionais. No episódio analisado, porém, o diesel apresentou maior variação percentual tanto no PPI quanto nos preços ao consumidor.

### Defasagem temporal

A análise de correlação com defasagens também fornece evidências de que as alterações no Brent e no PPI não se refletiram de maneira integral e contemporânea nos preços ao consumidor.

Para ambos os combustíveis, as maiores correlações entre as variações do Brent/PPI e as variações dos preços ao consumidor ocorreram com **uma semana de defasagem (lag 1)**.

No diesel, a correlação PPI -> preço ao consumidor atingiu aproximadamente **0,656 no lag 1**, enquanto Brent -> preço ao consumidor atingiu **0,470**.

Na gasolina, os valores correspondentes foram aproximadamente **0,479 para PPI -> preço ao consumidor** e **0,408 para Brent -> preço ao consumidor**.

As correlações indicam associação temporal entre as séries, mas não permitem estabelecer uma relação causal.

In [0]:
anp_silver_regional = spark.table("workspace.silver.precos_anp")

anp_silver_regional.printSchema()

root
 |-- data_coleta: date (nullable = true)
 |-- regiao: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- valor_venda: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- bandeira: string (nullable = true)
 |-- cnpj_revenda: string (nullable = true)
 |-- revenda: string (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- fonte: string (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col, date_trunc, avg, count, lit
)

anp_regiao_semanal = (
    anp_silver_regional
    .filter(
        (col("produto").isin("GASOLINA", "DIESEL S10")) &
        (col("data_coleta") >= lit(inicio_pre)) &
        (col("data_coleta") <= lit(fim_pos))
    )
    .withColumn(
        "semana",
        date_trunc("week", col("data_coleta")).cast("date")
    )
    .groupBy(
        "semana",
        "regiao",
        "produto"
    )
    .agg(
        avg("valor_venda").alias("preco_medio"),
        count("*").alias("observacoes")
    )
    .orderBy(
        "semana",
        "regiao",
        "produto"
    )
)

display(anp_regiao_semanal)

semana,regiao,produto,preco_medio,observacoes
2025-12-08,CO,DIESEL S10,6.094945652173912,184
2025-12-08,CO,GASOLINA,6.313478260869567,322
2025-12-08,N,DIESEL S10,6.498584070796461,226
2025-12-08,N,GASOLINA,6.662181208053694,298
2025-12-08,NE,DIESEL S10,6.017280966767378,662
2025-12-08,NE,GASOLINA,6.170251196172248,836
2025-12-08,S,DIESEL S10,6.13130801687764,474
2025-12-08,S,GASOLINA,6.285428973277074,711
2025-12-08,SE,DIESEL S10,6.099999999999976,1222
2025-12-08,SE,GASOLINA,6.079605885444004,1903


In [0]:
from pyspark.sql.functions import (
    col,
    when,
    avg,
    max as spark_max,
    round as spark_round
)

resumo_regional = (
    anp_regiao_semanal
    .withColumn(
        "periodo",
        when(
            col("semana") <= lit(fim_pre),
            "Pré-choque"
        ).otherwise("Pós-choque")
    )
    .groupBy(
        "regiao",
        "produto",
        "periodo"
    )
    .agg(
        avg("preco_medio").alias("preco_medio_periodo")
    )
)

comparacao_regional = (
    resumo_regional
    .groupBy("regiao", "produto")
    .pivot("periodo", ["Pré-choque", "Pós-choque"])
    .agg(spark_max("preco_medio_periodo"))
    .withColumn(
        "variacao_percentual",
        (
            (col("Pós-choque") - col("Pré-choque"))
            / col("Pré-choque")
        ) * 100
    )
    .select(
        "regiao",
        "produto",
        spark_round(col("Pré-choque"), 3).alias("media_pre"),
        spark_round(col("Pós-choque"), 3).alias("media_pos"),
        spark_round(
            col("variacao_percentual"), 2
        ).alias("variacao_percentual")
    )
    .orderBy("produto", "regiao")
)

display(comparacao_regional)

regiao,produto,media_pre,media_pos,variacao_percentual
CO,DIESEL S10,6.101,7.216,18.27
N,DIESEL S10,6.505,7.481,15.0
NE,DIESEL S10,6.063,7.281,20.07
S,DIESEL S10,6.14,7.238,17.89
SE,DIESEL S10,6.135,7.224,17.74
CO,GASOLINA,6.357,6.567,3.31
N,GASOLINA,6.713,7.165,6.72
NE,GASOLINA,6.296,6.939,10.22
S,GASOLINA,6.383,6.643,4.07
SE,GASOLINA,6.169,6.542,6.05


Databricks visualization. Run in Databricks to view.

In [0]:
# Parâmetros da análise do evento

inicio_pre = "2025-12-08"
fim_pre = "2026-02-23"

inicio_pos = "2026-03-02"
fim_pos = "2026-05-18"

# Carregamento da Silver da ANP
anp_silver_regional = spark.table("workspace.silver.precos_anp")

print("Parâmetros e tabela ANP carregados.")

Parâmetros e tabela ANP carregados.


In [0]:
from pyspark.sql.functions import (
    col,
    date_trunc,
    avg,
    lit,
    when,
    max as spark_max,
    round as spark_round
)

anp_silver_regional = spark.table("workspace.silver.precos_anp")

anp_uf_semanal = (
    anp_silver_regional
    .filter(
        (col("produto").isin("GASOLINA", "DIESEL S10")) &
        (col("data_coleta") >= lit(inicio_pre)) &
        (col("data_coleta") <= lit(fim_pos))
    )
    .withColumn(
        "semana",
        date_trunc("week", col("data_coleta")).cast("date")
    )
    .groupBy(
        "semana",
        "uf",
        "produto"
    )
    .agg(
        avg("valor_venda").alias("preco_medio")
    )
    .withColumn(
        "periodo",
        when(
            col("semana") <= lit(fim_pre),
            "Pré-choque"
        ).otherwise("Pós-choque")
    )
)

resumo_uf = (
    anp_uf_semanal
    .groupBy("uf", "produto", "periodo")
    .agg(
        avg("preco_medio").alias("preco_medio_periodo")
    )
)

comparacao_uf = (
    resumo_uf
    .groupBy("uf", "produto")
    .pivot("periodo", ["Pré-choque", "Pós-choque"])
    .agg(spark_max("preco_medio_periodo"))
    .withColumn(
        "variacao_percentual",
        (
            (col("Pós-choque") - col("Pré-choque"))
            / col("Pré-choque")
        ) * 100
    )
    .select(
        "uf",
        "produto",
        spark_round(col("Pré-choque"), 3).alias("media_pre"),
        spark_round(col("Pós-choque"), 3).alias("media_pos"),
        spark_round(
            col("variacao_percentual"), 2
        ).alias("variacao_percentual")
    )
    .orderBy("produto", col("variacao_percentual").desc())
)

display(comparacao_uf)

uf,produto,media_pre,media_pos,variacao_percentual
BA,DIESEL S10,6.209,7.902,27.26
TO,DIESEL S10,6.067,7.358,21.28
PR,DIESEL S10,6.057,7.319,20.84
DF,DIESEL S10,6.01,7.256,20.73
PI,DIESEL S10,6.062,7.307,20.53
SE,DIESEL S10,5.871,7.07,20.44
MA,DIESEL S10,6.01,7.186,19.56
GO,DIESEL S10,6.003,7.166,19.38
PE,DIESEL S10,5.87,6.997,19.2
SP,DIESEL S10,6.153,7.294,18.54


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

w_maiores = (
    Window
    .partitionBy("produto")
    .orderBy(col("variacao_percentual").desc())
)

w_menores = (
    Window
    .partitionBy("produto")
    .orderBy(col("variacao_percentual").asc())
)

maiores_uf = (
    comparacao_uf
    .withColumn("posicao", row_number().over(w_maiores))
    .filter(col("posicao") <= 5)
    .withColumn("grupo", lit("5 maiores variações"))
)

menores_uf = (
    comparacao_uf
    .withColumn("posicao", row_number().over(w_menores))
    .filter(col("posicao") <= 5)
    .withColumn("grupo", lit("5 menores variações"))
)

extremos_uf = (
    maiores_uf
    .unionByName(menores_uf)
    .select(
        "produto",
        "grupo",
        "posicao",
        "uf",
        "media_pre",
        "media_pos",
        "variacao_percentual"
    )
    .orderBy(
        "produto",
        "grupo",
        "posicao"
    )
)

display(extremos_uf)

produto,grupo,posicao,uf,media_pre,media_pos,variacao_percentual
DIESEL S10,5 maiores variações,1,BA,6.209,7.902,27.26
DIESEL S10,5 maiores variações,2,TO,6.067,7.358,21.28
DIESEL S10,5 maiores variações,3,PR,6.057,7.319,20.84
DIESEL S10,5 maiores variações,4,DF,6.01,7.256,20.73
DIESEL S10,5 maiores variações,5,PI,6.062,7.307,20.53
DIESEL S10,5 menores variações,1,AC,7.645,8.247,7.87
DIESEL S10,5 menores variações,2,AP,6.509,7.126,9.47
DIESEL S10,5 menores variações,3,AM,6.851,7.652,11.68
DIESEL S10,5 menores variações,4,ES,6.056,6.828,12.75
DIESEL S10,5 menores variações,5,RR,6.757,7.636,13.02


### Diferenças entre regiões e estados brasileiros

A análise dos preços ao consumidor indica que o comportamento após a data de referência não foi homogêneo entre as regiões e Unidades da Federação.

No nível regional, todas as cinco regiões apresentaram aumento dos preços médios de Diesel S10 e gasolina na comparação entre os períodos de 12 semanas anteriores e posteriores ao choque. Entretanto, a magnitude das variações foi diferente.

Para o Diesel S10, as variações regionais ficaram entre 15,00% no Norte e 20,07% no Nordeste. Para a gasolina, a dispersão foi maior proporcionalmente, com aumentos entre 3,31% no Centro-Oeste e 10,22% no Nordeste.

Quando a análise é aberta por UF, as diferenças ficam ainda maiores. No Diesel S10, as variações ficaram entre 7,87% no Acre e 27,26% na Bahia. Entre as cinco maiores variações também aparecem Tocantins (21,28%), Paraná (20,84%), Distrito Federal (20,73%) e Piauí (20,53%).

Na gasolina, as variações estaduais ficaram entre 1,09% no Distrito Federal e 12,85% na Bahia. Além da Bahia, as maiores variações foram observadas em Roraima (12,01%), Piauí (11,13%), Sergipe (10,58%) e Maranhão (9,85%).

Os resultados mostram, portanto, que a evolução dos preços após o conflito apresentou heterogeneidade territorial, tanto entre regiões quanto entre estados. A dispersão entre as UFs também demonstra que o comportamento regional agregado pode ocultar diferenças relevantes dentro de uma mesma região. O Distrito Federal ilustra como essas diferenças também variam por combustível: aparece entre as maiores altas do Diesel S10 (20,73%), mas teve a menor variação da gasolina (1,09%).


## Síntese dos resultados

Depois da data adotada como referência para a escalada do conflito no Oriente Médio, os preços de petróleo e combustíveis mudaram de patamar, mas não na mesma velocidade nem na mesma intensidade em todos os níveis da cadeia.

O choque apareceu primeiro no mercado internacional. O Brent teve média semanal de US$ 66,64 por barril nas 12 semanas anteriores ao evento e de US$ 110,28 nas 12 seguintes, alta de 65,48%. A subida foi rápida: a média semanal foi de US$ 71,36 na semana de 23/02 para US$ 85,28 em 02/03 e US$ 96,16 em 09/03. Os preços de paridade de importação (PPI) acompanharam o movimento, com alta de 73,15% no diesel e de 64,22% na gasolina.

No Brasil, o repasse ao consumidor foi bem mais suave. Entre as mesmas janelas, o Diesel S10 subiu 18,08% e a gasolina, 6,32%. A defasagem também aparece na análise semanal: na primeira semana inteira após a data de referência (início em 02/03), o Diesel S10 subiu só 0,73% e a gasolina, 0,11%. Na semana seguinte, as altas foram de 11,00% e 2,77%.

As correlações com defasagem apontam na mesma direção. Para os dois combustíveis, a associação mais forte entre as variações do Brent ou do PPI e as do preço ao consumidor ocorreu com uma semana de atraso (lag 1). Para o PPI, os coeficientes foram de cerca de 0,656 no diesel e 0,479 na gasolina; para o Brent, de 0,470 e 0,408.

O diesel reagiu mais que a gasolina no episódio, tanto no PPI quanto no consumidor. Na série histórica de 2018 a 2026, porém, a correlação contemporânea entre Brent e PPI foi maior para a gasolina (0,690) que para o diesel (0,588). Ou seja, qual combustível é "mais sensível" depende da métrica e da janela escolhidas.

Os preços ao consumidor também variaram muito entre regiões. No diesel, a alta foi de 15,00% no Norte a 20,07% no Nordeste; na gasolina, de 3,31% no Centro-Oeste a 10,22% no Nordeste. Por UF, a dispersão é maior: de 7,87% (Acre) a 27,26% (Bahia) no diesel e de 1,09% (Distrito Federal) a 12,85% (Bahia) na gasolina.

Em conjunto, os dados mostram uma sequência clara no período analisado: a mudança de patamar aparece primeiro no Brent e no PPI e, depois, de forma mais gradual e desigual, nos preços ao consumidor brasileiro. Essas análises, no entanto, são descritivas e exploratórias: as correlações e a ordem temporal não permitem atribuir a alta exclusivamente ao conflito, nem estimar uma taxa estrutural de repasse.

### Limitações e trabalhos futuros

O volume comercializado de combustíveis e o comportamento do etanol faziam parte do objetivo inicial, mas ficaram fora do escopo analítico desta versão do MVP. Incluí-los exigiria integrar novos conjuntos de dados ao pipeline, com etapas próprias de coleta, tratamento, modelagem e documentação.

Diante do escopo de um Produto Mínimo Viável, optou-se por concentrar o trabalho na construção de um fluxo completo e reprodutível envolvendo o Brent, os preços de paridade de importação e os preços dos combustíveis ao consumidor no Brasil.

A análise também apresenta limitações metodológicas. Trata-se de uma análise descritiva e correlacional, concentrada em um único evento e em uma janela pós-choque de 12 semanas. Por isso, os resultados não permitem atribuir causalidade nem estimar uma taxa de repasse entre os diferentes níveis de preços.

Como próximos passos, novas fontes poderão ser incorporadas para avaliar alterações no volume comercializado e comparar o comportamento do etanol com o de combustíveis mais diretamente relacionados ao petróleo. Também poderão ser incorporadas outras variáveis relacionadas à formação dos preços domésticos, permitindo avançar da análise descritiva e correlacional para modelos estatísticos mais abrangentes.